# 01 — Data Exploration

Load the arms control speech corpus, examine speech samples,
keyword distribution, and country/year coverage.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_config
cfg = load_config('../config.yaml')
print(cfg)

In [ ]:
# Load corpus
from src.data.load_ungdc import load_ungdc

corpus_df = load_ungdc(
    data_dir=str(cfg.data_dir),
    synthetic_mode=cfg.synthetic_mode,
    year_start=cfg.focus_start,
    year_end=cfg.focus_end,
)
print(f"Corpus: {len(corpus_df):,} rows")
print(f"Countries: {corpus_df['country_code'].nunique()}")
print(f"Years: {corpus_df['year'].min()} – {corpus_df['year'].max()}")
corpus_df.head()

In [ ]:
# Sample speeches
print("=== Sample speech segment (USA) ===")
usa = corpus_df[corpus_df['country_code'] == 'USA'].iloc[0]
print(f"Year: {usa['year']}")
print(usa['text'][:800])
print()
print("=== Sample speech segment (AUT - humanitarian coalition) ===")
aut = corpus_df[corpus_df['country_code'] == 'AUT'].iloc[0]
print(f"Year: {aut['year']}")
print(aut['text'][:800])

In [ ]:
# Country coverage heatmap
coverage = corpus_df.groupby(['country_code', 'year']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(coverage, cmap='Blues', ax=ax, cbar_kws={'label': 'N segments'})
ax.set_title('Speech Segment Coverage by Country-Year')
ax.set_xlabel('Year')
ax.set_ylabel('Country')
plt.tight_layout()
plt.show()

In [ ]:
# Keyword distribution
from src.data.segment import ALL_KEYWORDS, KEYWORDS_NUCLEAR, KEYWORDS_HUMANITARIAN

def count_keywords(texts, keywords):
    return sum(
        sum(1 for kw in keywords if kw in text.lower())
        for text in texts
    )

categories = {
    'Nuclear': KEYWORDS_NUCLEAR,
    'Humanitarian': KEYWORDS_HUMANITARIAN,
}

counts = {cat: count_keywords(corpus_df['text'], kws) for cat, kws in categories.items()}
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts.keys(), counts.values(), color=['#1f77b4', '#2ca02c'])
ax.set_ylabel('Total Keyword Hits')
ax.set_title('Keyword Frequency by Category')
plt.tight_layout()
plt.show()
print(counts)

In [ ]:
# Speeches per year
per_year = corpus_df.groupby('year').size()
fig, ax = plt.subplots(figsize=(12, 4))
per_year.plot(ax=ax, kind='bar', color='steelblue')
ax.set_xlabel('Year')
ax.set_ylabel('N Segments')
ax.set_title('Speech Segments per Year')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Preprocess and segment
from src.data.preprocess import preprocess_corpus
from src.data.segment import segment_by_keywords

clean_df = preprocess_corpus(corpus_df)
segs = segment_by_keywords(clean_df, window_size=3)
print(f"After segmentation: {len(segs):,} keyword-relevant segments")
print(f"Coverage: {segs['country_code'].nunique()} countries")
segs[['country_code', 'year', 'text', 'match_count']].head()